<a href="https://colab.research.google.com/github/Nitish-9k/CatandDogClassification-/blob/main/CatAndDogClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
import torchvision.transforms as transform

In [ ]:
# Import dataset from the kaggle

In [ ]:
!kaggle datasets download -d bhavikjikadara/dog-and-cat-classification-dataset
!unzip -q dog-and-cat-classification-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset
License(s): apache-2.0
100% 775M/775M [00:41<00:00, 19.6MB/s]



In [ ]:
!pip install -q kagglehub

import os
import kagglehub

path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")



Using Colab cache for faster access to the 'dog-and-cat-classification-dataset' dataset.


In [ ]:
print(path)
print(os.listdir(path))

/kaggle/input/dog-and-cat-classification-dataset
['PetImages']


In [ ]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader

data_root = path + "/PetImages"
print(data_root)

transform=transforms.Compose([
     transforms.Resize((64,64)),
     transforms.ToTensor(),
     transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
]
)

/kaggle/input/dog-and-cat-classification-dataset/PetImages


In [ ]:
dataset=datasets.ImageFolder(data_root,transform=transform)
from torch.utils.data import random_split
train_size=int(0.8*len(dataset))
test_size=len(dataset)-train_size
train_dataset,test_dataset = random_split(dataset, [train_size,test_size])
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

In [ ]:
from torch.nn.modules.pooling import MaxPool2d
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()
    self.conv_layer=nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)



    )

    self.fc_layer=nn.Sequential(
        nn.Linear(8*8*128,128),
        nn.ReLU(),
        nn.Linear(128,2)
    )

  def forward(self,x):
     x=self.conv_layer(x)
     x=x.view(x.size(0),-1)
     x=self.fc_layer(x)
     return x

In [ ]:
import torch.optim as optim
model=CNN()
criterion=nn.CrossEntropyLoss()
optim=optim.Adam(model.parameters())



In [ ]:
# model train
epochs=10

for epoch in range(epochs):
  model.train()
  epoch_trainig_loss=0
  for images,labels in train_loader:
    optim.zero_grad()
    output=model(images)

    loss=criterion(output,labels)
    loss.backward()
    optim.step()

    epoch_trainig_loss+=loss.item()
  print(f"for{epoch+1}/{epochs} trainig loss ={epoch_trainig_loss}")



/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


for1/10 trainig loss =376.79058825969696
for2/10 trainig loss =298.7349966466427
for3/10 trainig loss =253.9941709637642
for4/10 trainig loss =217.9930361956358
for5/10 trainig loss =185.07684634625912
for6/10 trainig loss =150.5872829258442
for7/10 trainig loss =116.89884453639388
for8/10 trainig loss =82.94609545916319
for9/10 trainig loss =58.158610401675105
for10/10 trainig loss =38.70641930238344


In [ ]:
correct_labels=0
total_labels=0
model.eval()
with torch.no_grad():
  for images,labels in test_loader:
    output=model(images)
    _,predict=torch.max(output,1)
    total_labels+=labels.size(0)
    correct_labels+=(predict==labels).sum().item()
print("accuracy = ",(correct_labels/total_labels)*100)

accuracy =  83.06


In [ ]:
print(total_labels)
print(correct_labels)

5000
4153
